In [ ]:
import sys
sys.path.append('..')

import copy
import inspect
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import importlib.util
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, confusion_matrix, ConfusionMatrixDisplay
from src.losses import ReconstructionPIT_MSELoss, ReconstructionPIT_NMSELoss
from src.dataset import create_train_val_datasets
from src.training import build_initial_state
from src.utils import get_device
from config import (
    small_dataset_path,
    medium_dataset_path,
    large_dataset_path,
    huge_dataset_path,
    TRAIN_SIZE,
    SPLIT_STRATEGY,
    SPLIT_STRATEGY,
    SEED_TRAIN,
    SEED_VAL,
    SPLIT_SEED,
    MAX_K,
    CONSTANT_K,
    BATCH_SIZE,
    NB_OF_STEPS
)

In [ ]:
device = get_device()
print(f"Using device : {device}") 

# run_name = Path("runs/run_20260722_171651") # SNR Weighting Run (Need to be on the branch "experiments/snr-weighting")
# run_name = Path("runs/run_20260722_171831") # Baseline Run  (Need to be on the branch "experiments/snr-weighting")
# run_name = Path("runs/run_20260722_171449") # Improved Model Run (Replaced LayerNorm by RMSNorm and FiLM) (Need to be on the branch "main" or "experiments/model_tuning")
# run_name = Path("runs/run_20260803_105358") # Best Model (Improved Model + Normalized MSE DB Loss + Pet only at t= 0) (Need to be on the branch "db-normalized-loss")
# run_name = Path("runs/run_20260813_201208") # Best Model For MAX_K = 5 (but which overfitted after) (Need to be on the branch "feature/variable-k-masking")
run_name = Path("runs/run_20260828_190432") # Best Model For MAX_K = 7 (Need to be on the branch "feature/variable-k-masking")

run_path = "../" / run_name / "checkpoints"
model_structure_path = run_path / "model_structure.py"
used_config_path = run_path / "used_config.py"
ckpt_path = run_path / "best_checkpoint.pth"
dataset_path = medium_dataset_path

spec = importlib.util.spec_from_file_location(
    "dynamic_model", model_structure_path
)
dynamic_model_module = importlib.util.module_from_spec(spec)
sys.modules["dynamic_model"] = dynamic_model_module
spec.loader.exec_module(dynamic_model_module)

config_spec = importlib.util.spec_from_file_location("run_config", used_config_path)
run_config = importlib.util.module_from_spec(config_spec)
config_spec.loader.exec_module(run_config)

In [ ]:
FlowSeparator = getattr(
    dynamic_model_module, "FlowSeparator"
)
model = FlowSeparator(max_k=run_config.MAX_K, n_blocks=run_config.N_BLOCKS).to(device)
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
number_of_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable parameters: {number_of_parameters}")
model.eval()
model_accepts_mask = "slot_mask" in inspect.signature(model.forward).parameters
def predict_velocity(X_t, t, mixture, K, slot_mask):
    if model_accepts_mask:
        return model(X_t, t, mixture, K, slot_mask)
    return model(X_t, t, mixture, K)

In [ ]:
NUMBER_OF_SAMPLES = 1000
BATCH_SIZE = 512
NB_OF_SAMPLING = 10

In [ ]:
_, val_dataset = create_train_val_datasets(
        dataset_path,
        train_size=run_config.TRAIN_SIZE,
        max_K=run_config.MAX_K,
        constant_K=run_config.CONSTANT_K,
        max_samples_train=0,
        max_samples_val=NUMBER_OF_SAMPLES,
        noise_train=True,
        noise_val=True,
        deterministic_train=False,
        deterministic_val=True,
        split_seed=run_config.SPLIT_SEED,
        split_strategy=run_config.SPLIT_STRATEGY,
        seed_train=run_config.SEED_TRAIN,
        seed_val=run_config.SEED_VAL,
        return_params=True,
)

In [ ]:
NB_OF_STEPS = run_config.NB_OF_STEPS
dt = 1.0 / NB_OF_STEPS
generator = torch.Generator(device=device)
generator.manual_seed(0)

dataloader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

criterion = ReconstructionPIT_MSELoss(residual_weight=1.0)
nmse = ReconstructionPIT_NMSELoss()

T = 0.5 * (1 - np.cos(np.pi * np.linspace(0.0, 1.0, NB_OF_STEPS + 1)))

In [ ]:
results_euler = []
total_loss_euler = 0.0
total_samples_euler = 0
generator.manual_seed(0)

with torch.no_grad():
    for mixture, X_1, K, slot_mask, params in tqdm(dataloader, desc="Evaluating using Euler solver", leave=False):
        mixture = mixture.to(device)
        X_1 = X_1.to(device)
        K = K.to(device)
        slot_mask = slot_mask.to(device)

        B = X_1.shape[0]
        effective_B = B * NB_OF_SAMPLING

        X_t = build_initial_state(
            mixture, X_1, K, slot_mask,
            nb_of_sampling=NB_OF_SAMPLING,
            generator=generator,
        )
        sampled_mixture = mixture.repeat_interleave(NB_OF_SAMPLING, dim=0)
        sampled_X_1 = X_1.repeat_interleave(NB_OF_SAMPLING, dim=0)
        sampled_K = K.repeat_interleave(NB_OF_SAMPLING, dim=0)
        sampled_slot_mask = slot_mask.repeat_interleave(NB_OF_SAMPLING, dim=0)

        for i in range(NB_OF_STEPS):
            t_val = float(T[i])
            dt = float(T[i + 1] - T[i])

            t = torch.full(
                (effective_B,),
                t_val,
                device=device,
                dtype=mixture.dtype,
            )

            v = predict_velocity(
                X_t, t, sampled_mixture, sampled_K, sampled_slot_mask
            )

            X_t = X_t + dt * v

        X_hat = X_t.reshape(B, NB_OF_SAMPLING, *X_t.shape[1:])

        flat_X_hat = X_hat.flatten(0, 1)
        loss, source_loss, residual_loss = criterion(
            flat_X_hat, sampled_X_1, sampled_slot_mask, reduction="none"
        )
        nmse_source, nmse_source_db = nmse(
            flat_X_hat[:, :-1],
            sampled_X_1[:, :-1],
            sampled_slot_mask[:, :-1],
            reduction="none",
        )
        loss = loss.reshape(B, NB_OF_SAMPLING)
        source_loss = source_loss.reshape(B, NB_OF_SAMPLING)
        residual_loss = residual_loss.reshape(B, NB_OF_SAMPLING)
        nmse_source = nmse_source.reshape(B, NB_OF_SAMPLING, -1)
        nmse_source_db = nmse_source_db.reshape(B, NB_OF_SAMPLING, -1)

        batch_mixture = mixture.cpu().numpy()
        batch_target = X_1.cpu().numpy()
        batch_prediction = X_hat.cpu().numpy()
        batch_K = K.cpu().numpy()
        batch_loss = loss.cpu().numpy()
        batch_source_loss = source_loss.cpu().numpy()
        batch_residual_loss = residual_loss.cpu().numpy()
        batch_nmse_source = nmse_source.cpu().numpy()
        batch_nmse_source_db = nmse_source_db.cpu().numpy()
        batch_snr = params["snr"].cpu().numpy()
        for b in range(B):
            results_euler.append({
                "mixture": batch_mixture[b],
                "target": batch_target[b],
                "prediction": batch_prediction[b],
                "K": int(batch_K[b]),
                "loss": batch_loss[b],
                "source_loss": batch_source_loss[b],
                "residual_loss": batch_residual_loss[b],
                "nmse_source": batch_nmse_source[b],
                "nmse_source_db": batch_nmse_source_db[b],
                "snr": batch_snr[b],
            })
        total_loss_euler += loss.sum().item()
        total_samples_euler += effective_B

average_loss_euler = total_loss_euler / total_samples_euler
print(f"Average loss using Euler solver: {average_loss_euler:.6f}")

In [ ]:
import pickle
from pathlib import Path

results_path = Path(
    f"results_{run_name.name}_euler_{NUMBER_OF_SAMPLES}_mixtures_"
    f"{NB_OF_SAMPLING}_samplings.pkl"
)
with results_path.open("wb") as f:
    pickle.dump(results_euler, f)
print(f"Saved {len(results_euler)} mixtures to {results_path}")

## Source reconstruction metrics by SNR

Each predicted source is first matched to a target source with Hungarian using the classic MSE. Its metrics are then assigned to the bin of the **true SNR of that target source**, not to a mixture SNR. The residual slot is not included here.

In [ ]:
from scipy.optimize import linear_sum_assignment


def compute_source_metrics(results, solver):
    rows = []

    for mixture_index, result in enumerate(results):
        target = np.asarray(result["target"])
        predictions = np.asarray(result["prediction"])
        if predictions.ndim == target.ndim:
            predictions = predictions[None]
        snrs = np.asarray(result["snr"]).reshape(-1)
        n_sources = int(result.get("K", target.shape[0] - 1))

        for sampling_index, prediction in enumerate(predictions):
            cost = np.mean(
                (prediction[:n_sources, None] - target[None, :n_sources]) ** 2,
                axis=(-1, -2),
            )
            predicted_slots, target_slots = linear_sum_assignment(cost)

            for predicted_slot, target_slot in zip(predicted_slots, target_slots):
                error = prediction[predicted_slot] - target[target_slot]
                error_energy = np.sum(error ** 2)
                source_energy = np.sum(target[target_slot] ** 2)
                source_nmse = error_energy / (source_energy + 1e-12)

                rows.append({
                    "solver": solver,
                    "mixture": mixture_index,
                    "sampling": sampling_index,
                    "source": int(target_slot),
                    "snr": float(snrs[target_slot]),
                    "mse": float(np.mean(error ** 2)),
                    "nmse": float(source_nmse),
                    "nmse_db": float(10 * np.log10(source_nmse + 1e-12)),
                })

    return pd.DataFrame(rows)


source_metrics = pd.concat([
    # compute_source_metrics(results_euler_simple_scheduler, "Euler Simple Scheduler"),
    compute_source_metrics(results_euler, "Euler"),
    # compute_source_metrics(results_heun, "Heun"),
], ignore_index=True)

source_metrics.head()

In [ ]:
snr_bin_edges = np.arange(10, 101, 10)
snr_bin_labels = [
    f"{left}-{right}"
    for left, right in zip(snr_bin_edges[:-1], snr_bin_edges[1:])
]

source_metrics["snr_bin"] = pd.cut(
    source_metrics["snr"],
    bins=snr_bin_edges,
    labels=snr_bin_labels,
    include_lowest=True,
)

snr_metrics = (
    source_metrics
    .groupby(["solver", "snr_bin"], observed=True)
    .agg(
        sources=("snr", "size"),
        mse=("mse", "mean"),
        nmse=("nmse", "mean"),
        nmse_db=("nmse_db", "mean"),
    )
    .reset_index()
)

snr_metrics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
plots = [
    ("mse", "MSE", "Mean source MSE"),
    ("nmse", "NMSE", "Mean source NMSE"),
    ("nmse_db", "NMSE (dB)", "Mean source NMSE in dB"),
]

for ax, (metric, ylabel, title) in zip(axes, plots):
    for solver, values in snr_metrics.groupby("solver", observed=True):
        ax.plot(
            values["snr_bin"].astype(str),
            values[metric],
            marker="o",
            label=solver,
        )

    ax.set_title(title)
    ax.set_xlabel("True source SNR")
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=45)
    ax.grid(alpha=0.3)
    ax.legend()

fig.tight_layout()
plt.show()

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pickle
from IPython.display import display
from scipy.optimize import linear_sum_assignment

path = "results_euler copy.pkl"

with open(path, "rb") as f:
    results_euler = pickle.load(f)


_result_sets = {
    name: values
    for name, values in {"Euler": results_euler}.items()
    if len(values) > 0
}
if not _result_sets:
    raise RuntimeError("Run the Euler or Heun evaluation cells first.")


def _align_prediction_with_target(prediction, target):
    """Align source slots with Hungarian; the residual stays last."""
    prediction = np.asarray(prediction)
    target = np.asarray(target)
    n_sources = target.shape[0] - 1

    cost = np.mean(
        (prediction[:n_sources, None] - target[None, :n_sources]) ** 2,
        axis=(-1, -2),
    )
    predicted_slots, target_slots = linear_sum_assignment(cost)

    aligned = np.empty_like(prediction)
    for predicted_slot, target_slot in zip(predicted_slots, target_slots):
        aligned[target_slot] = prediction[predicted_slot]
    aligned[-1] = prediction[-1]

    assignment = list(zip(predicted_slots.tolist(), target_slots.tolist()))
    return aligned, assignment


solver_widget = widgets.Dropdown(
    options=list(_result_sets),
    description="Results:",
)
element_widget = widgets.IntSlider(
    value=0,
    min=0,
    max=len(_result_sets[solver_widget.value]) - 1,
    step=1,
    description="Element:",
    continuous_update=False,
    layout=widgets.Layout(width="650px"),
)
previous_button = widgets.Button(description="◀ Previous")
next_button = widgets.Button(description="Next ▶")

search_widget = widgets.Text(
    placeholder="Enter element index to search",
    description="Search:",
    continuous_update=False,
    layout=widgets.Layout(width="650px"),
)


def _on_search_change(change):
    try:
        search_idx = int(change["new"])
        max_idx = len(_result_sets[solver_widget.value]) - 1
        if 0 <= search_idx <= max_idx:
            element_widget.value = search_idx
        else:
            print(f"Index out of range [0, {max_idx}]")
    except ValueError:
        if change["new"]:
            print("Please enter a valid integer")


def _update_element_range(change):
    element_widget.max = len(_result_sets[change["new"]]) - 1
    element_widget.value = 0


def _previous(_):
    element_widget.value = max(element_widget.min, element_widget.value - 1)


def _next(_):
    element_widget.value = min(element_widget.max, element_widget.value + 1)


solver_widget.observe(_update_element_range, names="value")
previous_button.on_click(_previous)
next_button.on_click(_next)


def _plot_reconstruction(solver, element):
    result = _result_sets[solver][element]

    target = np.asarray(result["target"])
    prediction = np.asarray(result["prediction"])
    k = int(result.get("K", target.shape[0] - 1))
    target = np.concatenate([target[:k], target[-1:]], axis=0)
    prediction = np.concatenate([prediction[:k], prediction[-1:]], axis=0)
    prediction, assignment = _align_prediction_with_target(
        prediction,
        target,
    )
    mixture = np.asarray(result["mixture"])

    n_slots, n_channels, n_frequency_bins = target.shape
    frequency_bins = np.arange(n_frequency_bins)
    source_snrs = np.asarray(result.get("snr", [])).reshape(-1)
    nmse_source = np.asarray(result.get("nmse_source", [])).reshape(-1)
    nmse_source_db = np.asarray(result.get("nmse_source_db", [])).reshape(-1)

    slot_names = [
        *[
            f"Source {source + 1} — SNR={source_snrs[source]:.2f}\nNMSE={nmse_source[source]:.6f} - NMSE(dB)={nmse_source_db[source]:.2f}"
            if source < len(source_snrs)
            else f"Source {source + 1} — SNR unavailable"
            for source in range(n_slots - 1)
        ],
        "Residual / noise",
    ]
    default_channel_names = ["A real", "A imag.", "E real", "E imag."]
    channel_names = [
        default_channel_names[channel]
        if channel < len(default_channel_names)
        else f"Channel {channel + 1}"
        for channel in range(n_channels)
    ]

    # Normalize mixture to shape (n_channels, n_frequency_bins)
    if mixture.ndim == 1:
        mixture = mixture[None, :]
    elif mixture.ndim > 2:
        mixture = mixture.reshape(mixture.shape[0], -1)

    if mixture.shape[0] != n_channels and mixture.shape[1] == n_channels:
        mixture = mixture.T

    if mixture.shape[0] != n_channels:
        raise ValueError(
            f"Expected mixture with {n_channels} channels, got shape {mixture.shape}"
        )

    mixture = mixture.reshape(n_channels, n_frequency_bins)

    # Récupération du min et max globaux de la mixture
    y_min = float(np.min(mixture))
    y_max = float(np.max(mixture))

    n_rows = n_slots + 1
    fig, axes = plt.subplots(
        n_rows,
        n_channels,
        figsize=(5 * n_channels, 3.0 * n_rows),
        sharex=True,
        squeeze=False,
    )

    color_mixture = "#38d667"
    color_target = "#0800ff"
    color_pred = "#ff8c00"

    # Plot mixture on the first row
    for channel in range(n_channels):
        ax = axes[0, channel]
        ax.plot(
            frequency_bins,
            mixture[channel],
            label="Mixture",
            linewidth=2,
            color=color_mixture,
        )
        ax.set_title(f"Mixture — {channel_names[channel]}")
        ax.grid(alpha=0.2)
        if channel == 0:
            ax.legend()

    # Plot targets and predictions on subsequent rows
    for slot in range(n_slots):
        for channel in range(n_channels):
            ax = axes[slot + 1, channel]

            ax.plot(
                frequency_bins,
                target[slot, channel],
                label="Target",
                linewidth=2.3,
                color=color_target,
                alpha=1.0,
            )
            ax.plot(
                frequency_bins,
                prediction[slot, channel],
                label="Prediction",
                linewidth=1.3,
                linestyle="--",
                color=color_pred,
                alpha=1.0,
            )

            ax.set_title(f"{slot_names[slot]} — {channel_names[channel]}")
            ax.grid(alpha=0.2)
            if slot == n_slots - 1:
                ax.set_xlabel("Frequency bin")
            if slot == 0 and channel == 0:
                ax.legend()

    # Application des limites y_min et y_max à l'ensemble des sous-graphiques
    for ax in axes.flat:
        ax.set_ylim(y_min, y_max)

    stored_loss = float(result.get("loss", np.nan))
    stored_source_loss = float(result.get("source_loss", np.nan))
    stored_residual_loss = float(result.get("residual_loss", np.nan))

    fig.suptitle(
        f"{solver} — element {element}/{len(_result_sets[solver]) - 1}"
        f" — loss={stored_loss:.6g}"
        f" — source loss={stored_source_loss:.6g}"
        f" — residual loss={stored_residual_loss:.6g}"
        f" — predicted→target PIT: {assignment}"
        f"\n",
        fontsize=14,
    )
    fig.tight_layout()
    plt.show()


search_widget.observe(_on_search_change, names="value")

controls = widgets.VBox([
    widgets.HBox([solver_widget, previous_button, next_button]),
    search_widget,
    element_widget,
])
plots = widgets.interactive_output(
    _plot_reconstruction,
    {"solver": solver_widget, "element": element_widget},
)
display(controls, plots)

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pickle

from IPython.display import display
from scipy.optimize import linear_sum_assignment


path = "results_run_20260828_190432_euler_1000_mixtures_10_samplings.pkl"

with open(path, "rb") as f:
    results_euler = pickle.load(f)


NB_OF_SAMPLING = 10

_result_sets = {
    name: values
    for name, values in {
        "Euler": results_euler
    }.items()
    if len(values) > 0
}

if not _result_sets:
    raise RuntimeError(
        "Run the Euler or Heun evaluation cells first."
    )


def _align_prediction_with_target(prediction, target):
    """Align source slots with Hungarian; the residual stays last."""

    prediction = np.asarray(prediction)
    target = np.asarray(target)

    n_sources = target.shape[0] - 1

    cost = np.mean(
        (
            prediction[:n_sources, None]
            - target[None, :n_sources]
        ) ** 2,
        axis=(-1, -2),
    )

    predicted_slots, target_slots = (
        linear_sum_assignment(cost)
    )

    aligned = np.empty_like(prediction)

    for predicted_slot, target_slot in zip(
        predicted_slots,
        target_slots,
    ):
        aligned[target_slot] = (
            prediction[predicted_slot]
        )

    aligned[-1] = prediction[-1]

    assignment = list(
        zip(
            predicted_slots.tolist(),
            target_slots.tolist(),
        )
    )

    return aligned, assignment


def _source_uncertainty(
    aligned_predictions,
    eps=1e-12,
):
    """
    Compute uncertainty across the different
    Flow Matching samplings.

    Returns
    -------
    uncertainty_rms : (K,)
        Absolute RMS uncertainty for each source.

    relative_uncertainty : (K,)
        Relative uncertainty U_k.

        U_k =
            RMS(sample disagreement)
            ------------------------
            RMS(sample amplitude)
    """

    aligned_predictions = np.asarray(
        aligned_predictions
    )

    # Keep sources only, remove residual
    source_predictions = (
        aligned_predictions[:, :-1]
    )

    # Mean reconstruction across samplings
    consensus = source_predictions.mean(axis=0)

    # ---------------------------------------------------------
    # Absolute uncertainty
    # ---------------------------------------------------------

    # std at every channel / frequency bin
    pointwise_std = source_predictions.std(
        axis=0
    )

    # Collapse (C, F) into one typical ± sigma
    # per source
    uncertainty_rms = np.sqrt(
        np.mean(
            pointwise_std ** 2,
            axis=(1, 2),
        )
    )

    # ---------------------------------------------------------
    # Relative uncertainty U_k
    # ---------------------------------------------------------

    # disagreement between stochastic
    # reconstructions
    dispersion_rms = np.sqrt(
        np.mean(
            (
                source_predictions
                - consensus[None]
            ) ** 2,
            axis=(0, 2, 3),
        )
    )

    # typical amplitude of the stochastic
    # reconstructions
    sample_rms = np.sqrt(
        np.mean(
            source_predictions ** 2,
            axis=(0, 2, 3),
        )
    )

    relative_uncertainty = (
        dispersion_rms
        / (sample_rms + eps)
    )

    return (
        uncertainty_rms,
        relative_uncertainty,
    )


solver_widget = widgets.Dropdown(
    options=list(_result_sets),
    description="Results:",
)

element_widget = widgets.IntSlider(
    value=0,
    min=0,
    max=len(
        _result_sets[solver_widget.value]
    ) - 1,
    step=1,
    description="Element:",
    continuous_update=False,
    layout=widgets.Layout(
        width="650px"
    ),
)

view_widget = widgets.ToggleButtons(
    options=[
        ("Single sample", "sample"),
        ("Mean ± std", "mean"),
    ],
    description="View:",
)

sampling_widget = widgets.IntSlider(
    value=0,
    min=0,
    max=NB_OF_SAMPLING - 1,
    step=1,
    description="Sampling:",
    continuous_update=False,
    layout=widgets.Layout(
        width="650px"
    ),
)

previous_button = widgets.Button(
    description="◀ Previous"
)

next_button = widgets.Button(
    description="Next ▶"
)

search_widget = widgets.Text(
    placeholder=(
        "Enter element index to search"
    ),
    description="Search:",
    continuous_update=False,
    layout=widgets.Layout(
        width="650px"
    ),
)


def _on_search_change(change):
    try:
        search_idx = int(change["new"])

        max_idx = (
            len(
                _result_sets[
                    solver_widget.value
                ]
            )
            - 1
        )

        if 0 <= search_idx <= max_idx:
            element_widget.value = search_idx
        else:
            print(
                f"Index out of range "
                f"[0, {max_idx}]"
            )

    except ValueError:
        if change["new"]:
            print(
                "Please enter a valid integer"
            )


def _update_element_range(change):
    element_widget.max = (
        len(_result_sets[change["new"]])
        - 1
    )

    element_widget.value = 0

    _update_sampling_range()


def _update_sampling_range(*_):
    result = _result_sets[
        solver_widget.value
    ][element_widget.value]

    predictions = np.asarray(
        result["prediction"]
    )

    target = np.asarray(
        result["target"]
    )

    sampling_count = (
        1
        if predictions.ndim == target.ndim
        else predictions.shape[0]
    )

    sampling_widget.max = (
        sampling_count - 1
    )

    sampling_widget.value = min(
        sampling_widget.value,
        sampling_widget.max,
    )


def _previous(_):
    element_widget.value = max(
        element_widget.min,
        element_widget.value - 1,
    )


def _next(_):
    element_widget.value = min(
        element_widget.max,
        element_widget.value + 1,
    )


solver_widget.observe(
    _update_element_range,
    names="value",
)

element_widget.observe(
    _update_sampling_range,
    names="value",
)

previous_button.on_click(_previous)
next_button.on_click(_next)


def _plot_reconstruction(
    solver,
    element,
    view,
    sampling,
):
    result = _result_sets[
        solver
    ][element]

    target = np.asarray(
        result["target"]
    )

    predictions = np.asarray(
        result["prediction"]
    )

    if predictions.ndim == target.ndim:
        predictions = predictions[None]

    k = int(
        result.get(
            "K",
            target.shape[0] - 1,
        )
    )

    # Keep K active sources + residual
    target = np.concatenate(
        [
            target[:k],
            target[-1:],
        ],
        axis=0,
    )

    # ---------------------------------------------------------
    # Align every sampling with the target
    # ---------------------------------------------------------

    aligned_predictions = []
    assignments = []

    for current_prediction in predictions:
        current_prediction = np.concatenate(
            [
                current_prediction[:k],
                current_prediction[-1:],
            ],
            axis=0,
        )

        aligned, assignment = (
            _align_prediction_with_target(
                current_prediction,
                target,
            )
        )

        aligned_predictions.append(
            aligned
        )

        assignments.append(
            assignment
        )

    aligned_predictions = np.stack(
        aligned_predictions
    )

    # ---------------------------------------------------------
    # Uncertainty from ALL samplings
    # ---------------------------------------------------------

    (
        uncertainty_rms,
        relative_uncertainty,
    ) = _source_uncertainty(
        aligned_predictions
    )

    # ---------------------------------------------------------
    # Display mode
    # ---------------------------------------------------------

    if view == "mean":
        prediction = (
            aligned_predictions.mean(
                axis=0
            )
        )

        # std includes sources + residual
        # shape = (K+1, C, F)
        prediction_std = (
            aligned_predictions.std(
                axis=0
            )
        )

        assignment_label = (
            f"{len(aligned_predictions)} "
            f"aligned samples"
        )

    else:
        sampling = min(
            sampling,
            len(aligned_predictions) - 1,
        )

        prediction = (
            aligned_predictions[sampling]
        )

        prediction_std = None

        assignment_label = (
            assignments[sampling]
        )

    mixture = np.asarray(
        result["mixture"]
    )

    (
        n_slots,
        n_channels,
        n_frequency_bins,
    ) = target.shape

    frequency_bins = np.arange(
        n_frequency_bins
    )

    source_snrs = np.asarray(
        result.get("snr", [])
    ).reshape(-1)

    # ---------------------------------------------------------
    # NMSE
    # ---------------------------------------------------------

    source_error_energy = np.sum(
        (
            prediction[:-1]
            - target[:-1]
        ) ** 2,
        axis=(-1, -2),
    )

    source_energy = np.sum(
        target[:-1] ** 2,
        axis=(-1, -2),
    )

    nmse_source = (
        source_error_energy
        / (source_energy + 1e-12)
    )

    nmse_source_db = (
        10
        * np.log10(
            nmse_source + 1e-12
        )
    )

    # ---------------------------------------------------------
    # Titles
    # ---------------------------------------------------------

    slot_names = [
        *[
            (
                f"Source {source + 1} — "
                f"SNR="
                f"{source_snrs[source]:.2f} — "
                f"±σ="
                f"{uncertainty_rms[source]:.4f} — "
                f"U="
                f"{100 * relative_uncertainty[source]:.2f}%\n"
                f"NMSE="
                f"{nmse_source[source]:.6f} — "
                f"NMSE(dB)="
                f"{nmse_source_db[source]:.2f}"
            )
            if source < len(source_snrs)
            else
            (
                f"Source {source + 1} — "
                f"±σ="
                f"{uncertainty_rms[source]:.4f} — "
                f"U="
                f"{100 * relative_uncertainty[source]:.2f}%"
            )
            for source
            in range(n_slots - 1)
        ],
        "Residual / noise",
    ]

    default_channel_names = [
        "A real",
        "A imag.",
        "E real",
        "E imag.",
    ]

    channel_names = [
        (
            default_channel_names[channel]
            if channel
            < len(default_channel_names)
            else
            f"Channel {channel + 1}"
        )
        for channel
        in range(n_channels)
    ]

    # ---------------------------------------------------------
    # Normalize mixture shape
    # ---------------------------------------------------------

    if mixture.ndim == 1:
        mixture = mixture[None, :]

    elif mixture.ndim > 2:
        mixture = mixture.reshape(
            mixture.shape[0],
            -1,
        )

    if (
        mixture.shape[0] != n_channels
        and mixture.shape[1]
        == n_channels
    ):
        mixture = mixture.T

    if mixture.shape[0] != n_channels:
        raise ValueError(
            f"Expected mixture with "
            f"{n_channels} channels, "
            f"got shape {mixture.shape}"
        )

    mixture = mixture.reshape(
        n_channels,
        n_frequency_bins,
    )

    # ---------------------------------------------------------
    # Global limits from mixture
    # ---------------------------------------------------------

    y_min = float(
        np.min(mixture)
    )

    y_max = float(
        np.max(mixture)
    )

    # Mixture:
    #   1 row
    #
    # Each source / residual:
    #   1 global-scale row
    #   1 local-scale row
    n_rows = (
        1
        + 2 * n_slots
    )

    fig, axes = plt.subplots(
        n_rows,
        n_channels,
        figsize=(
            5 * n_channels,
            3.0 * n_rows,
        ),
        sharex=True,
        squeeze=False,
    )

    color_mixture = "#38d667"
    color_target = "#0800ff"
    color_pred = "#ff8c00"

    # ---------------------------------------------------------
    # Mixture
    # ---------------------------------------------------------

    for channel in range(n_channels):
        ax = axes[0, channel]

        ax.plot(
            frequency_bins,
            mixture[channel],
            label="Mixture",
            linewidth=2,
            color=color_mixture,
        )

        ax.set_title(
            f"Mixture — "
            f"{channel_names[channel]}"
        )

        ax.set_ylim(
            y_min,
            y_max,
        )

        ax.grid(alpha=0.2)

        if channel == 0:
            ax.legend()

    # ---------------------------------------------------------
    # Sources + residual
    #
    # For each slot:
    #
    # row 1 -> global mixture scale
    # row 2 -> local source scale
    # ---------------------------------------------------------

    for slot in range(n_slots):

        global_row = (
            1
            + 2 * slot
        )

        local_row = (
            global_row + 1
        )

        # -----------------------------------------------------
        # Local y limits
        #
        # They are calculated over ALL channels of the
        # current source so A real / A imag / E real / E imag
        # still use the same y scale.
        # -----------------------------------------------------

        local_values = [
            target[
                slot
            ].reshape(-1),

            prediction[
                slot
            ].reshape(-1),
        ]

        # Include uncertainty envelope in the
        # local limits
        if prediction_std is not None:
            local_values.extend(
                [
                    (
                        prediction[slot]
                        - prediction_std[slot]
                    ).reshape(-1),

                    (
                        prediction[slot]
                        + prediction_std[slot]
                    ).reshape(-1),
                ]
            )

        local_values = np.concatenate(
            local_values
        )

        local_y_min = float(
            np.min(local_values)
        )

        local_y_max = float(
            np.max(local_values)
        )

        # Small margin around the local plot
        local_range = (
            local_y_max
            - local_y_min
        )

        if local_range <= 1e-12:
            local_scale = max(
                abs(local_y_min),
                abs(local_y_max),
                1.0,
            )

            local_padding = (
                0.05
                * local_scale
            )

        else:
            local_padding = (
                0.05
                * local_range
            )

        local_y_min -= local_padding
        local_y_max += local_padding

        # -----------------------------------------------------
        # Plot global + local versions
        # -----------------------------------------------------

        for channel in range(n_channels):

            for row, scale_label in [
                (
                    global_row,
                    "\nglobal scale",
                ),
                (
                    local_row,
                    "\nlocal scale",
                ),
            ]:

                ax = axes[
                    row,
                    channel,
                ]

                # Target
                ax.plot(
                    frequency_bins,
                    target[
                        slot,
                        channel,
                    ],
                    label="Target",
                    linewidth=2.3,
                    color=color_target,
                    alpha=1.0,
                )

                # Prediction
                ax.plot(
                    frequency_bins,
                    prediction[
                        slot,
                        channel,
                    ],
                    label="Prediction",
                    linewidth=1.3,
                    linestyle="--",
                    color=color_pred,
                    alpha=1.0,
                )

                # Mean ± std
                if prediction_std is not None:
                    ax.fill_between(
                        frequency_bins,

                        prediction[
                            slot,
                            channel,
                        ]
                        - prediction_std[
                            slot,
                            channel,
                        ],

                        prediction[
                            slot,
                            channel,
                        ]
                        + prediction_std[
                            slot,
                            channel,
                        ],

                        color=color_pred,
                        alpha=0.2,
                        label="± 1 std",
                    )

                ax.set_title(
                    f"{slot_names[slot]} — "
                    f"{channel_names[channel]} — "
                    f"{scale_label}"
                )

                # -----------------------------
                # Global / local y scale
                # -----------------------------

                if row == global_row:
                    ax.set_ylim(
                        y_min,
                        y_max,
                    )

                else:
                    ax.set_ylim(
                        local_y_min,
                        local_y_max,
                    )

                ax.grid(
                    alpha=0.2
                )

                # x label only at the very
                # bottom of the whole figure
                if (
                    slot == n_slots - 1
                    and row == local_row
                ):
                    ax.set_xlabel(
                        "Frequency bin"
                    )

                if (
                    slot == 0
                    and channel == 0
                ):
                    ax.legend()

    # ---------------------------------------------------------
    # Losses
    # ---------------------------------------------------------

    stored_source_loss = float(
        np.mean(
            (
                prediction[:-1]
                - target[:-1]
            ) ** 2
        )
    )

    stored_residual_loss = float(
        np.mean(
            (
                prediction[-1]
                - target[-1]
            ) ** 2
        )
    )

    stored_loss = (
        stored_source_loss
        + stored_residual_loss
    )

    view_label = (
        f"sample {sampling}"
        if view == "sample"
        else "sampling mean ± std"
    )

    fig.suptitle(
        f"{solver} — "
        f"element {element}/"
        f"{len(_result_sets[solver]) - 1}"
        f" — loss={stored_loss:.6g}"
        f" — source loss="
        f"{stored_source_loss:.6g}"
        f" — residual loss="
        f"{stored_residual_loss:.6g}"
        f" — {view_label}"
        f" — predicted→target PIT: "
        f"{assignment_label}"
        f"\n\n\n",
        fontsize=14,
    )

    fig.tight_layout()

    display(fig)

    plt.close(fig)


search_widget.observe(
    _on_search_change,
    names="value",
)

controls = widgets.VBox(
    [
        widgets.HBox(
            [
                solver_widget,
                previous_button,
                next_button,
            ]
        ),
        search_widget,
        element_widget,
        view_widget,
        sampling_widget,
    ]
)

plt.close("all")

plots = widgets.interactive_output(
    _plot_reconstruction,
    {
        "solver": solver_widget,
        "element": element_widget,
        "view": view_widget,
        "sampling": sampling_widget,
    },
)

display(
    controls,
    plots,
)

In [ ]:
import importlib.util
import pickle
import sys
from pathlib import Path

import h5py
import numpy as np
from scipy.optimize import linear_sum_assignment


# ------------------------------------------------------------
# Localisation du dépôt
# ------------------------------------------------------------

repo_root = next(
    (
        path
        for path in (Path.cwd(), Path.cwd().parent)
        if (path / "src" / "dataset.py").exists()
    ),
    None,
)

if repo_root is None:
    raise RuntimeError("Impossible de localiser la racine du dépôt.")

sys.path.insert(0, str(repo_root))

from src.dataset import create_train_val_datasets


# ------------------------------------------------------------
# Cas sélectionnés
#
# element_index : index 0-based dans les 1000 mixtures
# source_number : numéro 1-based affiché par le widget
# quality       : 1=mauvais, 2=moyen, 3=très bien
# ------------------------------------------------------------

cases = [
    # SNR bas
    (0,   1, 10.80, 1),
    (306, 1, 13.90, 2),
    (51,  5, 22.47, 3),

    # SNR moyen
    (51,  4, 30.54, 1),
    (51,  1, 40.43, 2),
    (927, 2, 35.51, 3),

    # SNR haut
    (65,  5, 75.94, 1),
    (394, 4, 50.81, 2),
    (468, 3, 88.19, 3),
]


# ------------------------------------------------------------
# Chargement du pickle
# ------------------------------------------------------------

results_path = (
    repo_root
    / "notebooks"
    / (
        "results_run_20260828_190432_euler_"
        "1000_mixtures_10_samplings.pkl"
    )
)

with results_path.open("rb") as file:
    results = pickle.load(file)

if len(results) != 1000:
    raise ValueError(
        f"Le pickle contient {len(results)} éléments au lieu de 1000."
    )


# ------------------------------------------------------------
# Chargement de la configuration exacte de la run
# ------------------------------------------------------------

config_path = (
    repo_root
    / "runs"
    / "run_20260828_190432"
    / "checkpoints"
    / "used_config.py"
)

spec = importlib.util.spec_from_file_location(
    "run_20260828_config",
    config_path,
)

run_config = importlib.util.module_from_spec(spec)
spec.loader.exec_module(run_config)


# ------------------------------------------------------------
# Reconstruction exacte du dataset de validation
#
# Le pickle avait été produit avec medium_dataset_path,
# shuffle=False et return_params=True.
# ------------------------------------------------------------

_, val_dataset = create_train_val_datasets(
    dataset_path=run_config.medium_dataset_path,
    train_size=run_config.TRAIN_SIZE,
    max_K=run_config.MAX_K,
    constant_K=run_config.CONSTANT_K,
    max_samples_train=0,
    max_samples_val=1000,
    noise_train=True,
    noise_val=True,
    deterministic_train=False,
    deterministic_val=True,
    split_seed=run_config.SPLIT_SEED,
    split_strategy=run_config.SPLIT_STRATEGY,
    seed_train=run_config.SEED_TRAIN,
    seed_val=run_config.SEED_VAL,
    return_params=True,
)


# ------------------------------------------------------------
# Conteneurs de sortie
# ------------------------------------------------------------

output = {
    "element_index": [],
    "source_number": [],
    "source_reconstructed": [],
    "source_real": [],
    "snr": [],
    "reconstruction_quality": [],
    "residual_reconstructed": [],
    "residual_real": [],
}

# Paramètres physiques réellement présents sous params/ dans
# le dataset original.
parameter_names = [
    name
    for name in val_dataset.attr_names
    if name != "snr"
]

parameter_values = {
    name: []
    for name in parameter_names
}


# ------------------------------------------------------------
# Extraction des neuf cas
# ------------------------------------------------------------

for (
    element_index,
    source_number,
    expected_snr,
    quality,
) in cases:
    source_slot = source_number - 1

    result = results[element_index]

    (
        mixture,
        true_slots,
        k,
        slot_mask,
        params,
    ) = val_dataset[element_index]

    k = int(k)
    true_slots = np.asarray(true_slots)

    # Vérifie que la régénération correspond exactement au pickle.
    if k != int(result["K"]):
        raise RuntimeError(
            f"K différent pour l’élément {element_index}."
        )

    np.testing.assert_array_equal(
        mixture,
        np.asarray(result["mixture"]),
        err_msg=(
            f"Mixture différente pour l’élément {element_index}. "
            "Les seeds ou la configuration ne correspondent pas."
        ),
    )

    np.testing.assert_array_equal(
        true_slots,
        np.asarray(result["target"]),
        err_msg=(
            f"Target différent pour l’élément {element_index}."
        ),
    )

    if source_slot >= k:
        raise IndexError(
            f"L’élément {element_index} contient {k} sources, "
            f"mais la source {source_number} a été demandée."
        )

    true_snr = float(params["snr"][source_slot])

    if not np.isclose(true_snr, expected_snr, atol=0.01):
        raise RuntimeError(
            f"SNR inattendu pour élément={element_index}, "
            f"source={source_number}: "
            f"{true_snr:.4f} au lieu de {expected_snr:.2f}."
        )

    # --------------------------------------------------------
    # Premier sampling uniquement
    # --------------------------------------------------------

    predictions = np.asarray(result["prediction"])

    if predictions.ndim == true_slots.ndim:
        first_prediction = predictions
    else:
        first_prediction = predictions[0]

    # --------------------------------------------------------
    # Hungarian entre les K sources prédites et réelles
    # --------------------------------------------------------

    cost = np.mean(
        (
            first_prediction[:k, None]
            - true_slots[None, :k]
        ) ** 2,
        axis=(-1, -2),
    )

    predicted_slots, target_slots = (
        linear_sum_assignment(cost)
    )

    # Réordonne les prédictions selon l’ordre des vraies sources.
    aligned_sources = np.empty_like(
        first_prediction[:k]
    )

    aligned_sources[target_slots] = (
        first_prediction[predicted_slots]
    )

    output["element_index"].append(element_index)
    output["source_number"].append(source_number)
    output["source_reconstructed"].append(
        aligned_sources[source_slot]
    )
    output["source_real"].append(
        true_slots[source_slot]
    )
    output["snr"].append(true_snr)
    output["reconstruction_quality"].append(quality)

    # Le résidu est toujours le dernier slot et ne participe
    # pas au Hungarian.
    output["residual_reconstructed"].append(
        first_prediction[-1]
    )
    output["residual_real"].append(
        true_slots[-1]
    )

    # Paramètres physiques de la vraie source sélectionnée.
    for parameter_name in parameter_names:
        parameter_values[parameter_name].append(
            np.asarray(params[parameter_name])[source_slot]
        )


# ------------------------------------------------------------
# Création du fichier HDF5
# ------------------------------------------------------------

output_path = (
    repo_root
    / "notebooks"
    / "selected_reconstruction_cases.h5"
)

with h5py.File(output_path, "w") as h5_file:
    h5_file.attrs["description"] = (
        "Nine manually selected reconstruction examples "
        "from run_20260828_190432."
    )
    h5_file.attrs["sampling_index"] = 0
    h5_file.attrs["element_indexing"] = "0-based"
    h5_file.attrs["source_numbering"] = "1-based"
    h5_file.attrs["reconstruction_quality"] = (
        "1=bad, 2=medium, 3=very good"
    )
    h5_file.attrs["results_file"] = results_path.name
    h5_file.attrs["dataset_path"] = str(
        run_config.medium_dataset_path
    )

    h5_file.create_dataset(
        "element_index",
        data=np.asarray(
            output["element_index"],
            dtype=np.int32,
        ),
    )

    h5_file.create_dataset(
        "source_number",
        data=np.asarray(
            output["source_number"],
            dtype=np.int16,
        ),
    )

    h5_file.create_dataset(
        "source_reconstructed",
        data=np.stack(output["source_reconstructed"]),
        compression="gzip",
    )

    h5_file.create_dataset(
        "source_real",
        data=np.stack(output["source_real"]),
        compression="gzip",
    )

    h5_file.create_dataset(
        "snr",
        data=np.asarray(output["snr"], dtype=np.float32),
    )

    h5_file.create_dataset(
        "reconstruction_quality",
        data=np.asarray(
            output["reconstruction_quality"],
            dtype=np.int8,
        ),
    )

    h5_file.create_dataset(
        "residual_reconstructed",
        data=np.stack(output["residual_reconstructed"]),
        compression="gzip",
    )

    h5_file.create_dataset(
        "residual_real",
        data=np.stack(output["residual_real"]),
        compression="gzip",
    )

    # Un dataset HDF5 distinct pour chaque paramètre physique.
    for parameter_name, values in parameter_values.items():
        h5_file.create_dataset(
            parameter_name,
            data=np.stack(values),
        )


# ------------------------------------------------------------
# Résumé
# ------------------------------------------------------------

print(f"Fichier créé : {output_path}")
print("\nContenu :")

with h5py.File(output_path, "r") as h5_file:
    for name, dataset in h5_file.items():
        print(
            f"  {name:28s} "
            f"shape={dataset.shape}, dtype={dataset.dtype}"
        )